In [7]:
import pandas as pd
import pickle
import tomllib
import plotly.graph_objects as go
import polars as pl
pd.set_option('display.expand_frame_repr', False)

In [11]:
with open("config.toml", "rb") as f:
    config = tomllib.load(f)

for k in config:
    print(k)

generate_timeseries


# Disaggregation evaluation

## Load Disaggregator temporal disaggregations
Some csv files are very big. The dataframe for CTS is around 16GB. Make sure to have enough memory if you want to load all the file. If you want to select specific columns, regions, and industry sector, using polars *scan_csv* method may be a better option. Here we use the pickle files.

In [ ]:
# year of comparison
year = 2024

In [ ]:
%%time
format = "pkl"
path = "/mnt/data/oe215/rhindrikson/el_load/demandregio_disaggregation/timezone_for_all"
hh_df =  pd.read_pickle(path + f"/households/temporal_disaggregation_households_power_slp_{year}.{format}")
ind_df = pd.read_pickle(path + f"/industry/temporal_disaggregation_power_industry_{year}.{format}")
cts_df = pd.read_pickle(path + f"/cts/temporal_disaggregation_power_cts_{year}.{format}")

In [ ]:
hh_total = hh_df.sum().sum()
ind_total = ind_df.sum().sum()
cts_total = cts_df.sum().sum()
total_sum = hh_total + ind_total + cts_total
print("HH total TWh: ", hh_total/1e6)
print("Industry total TWh: ", ind_total/1e6)
print("CTS total TWh: ", cts_total/1e6)
print("Sum of all: ", (total_sum)/1e6)

In [ ]:
# Sum across all industry sectors and regions
ind_sum_df = ind_df.sum(axis=1).to_frame(name="total")
cts_sum_df = cts_df.sum(axis=1).to_frame(name="total")
hh_sum_df = hh_df.sum(axis=1).to_frame(name="total")

In [ ]:
print("industry_sum index type:", type(ind_sum_df.index))
print("cts_sum index type:", type(cts_sum_df.index))
print("households_sum index type:", type(hh_sum_df.index))

In [ ]:
# Convert all dataframe indices to datetime before concatenating
ind_sum_df.index = pd.to_datetime(ind_sum_df.index)
cts_sum_df.index = pd.to_datetime(cts_sum_df.index)
hh_sum_df.index = pd.to_datetime(hh_sum_df.index)

In [ ]:
if len(ind_sum_df) != len(cts_sum_df) != len(hh_df):
    print("WARNING: 15 minutes interval do not match")
print(ind_sum_df.head(2))
print(cts_sum_df.head(2))
print(hh_sum_df.head(2))

In [ ]:
df = pd.concat([ind_sum_df, cts_sum_df, hh_sum_df], axis=1).sum(axis=1).to_frame(name='total').round(2)
df = df.sort_index()
# Check if there are any duplicates
has_duplicates = df.index.duplicated().any()
print(f"Has duplicates: {has_duplicates}")
print(df.head())

In [ ]:
# check if there is NaN values
if df.isna().any().any():
    print("WARNING: NaN values.")

## Load ENTSOE time series

In [ ]:
ent_df = pd.read_csv("/mnt/data/oe215/rhindrikson/datasets/load/entsoe-15min/training_data_21_24/data.csv")
ent_df["ds"] = pd.to_datetime(ent_df["ds"])
ent_df_year = ent_df[ent_df["ds"].dt.year == year].copy()
# Here I divided by 4 because. It seems like the values are give the hourly MWh consumption. To get the actual value of MW we need to divide it by 4.
ent_df_year["y"] = ent_df_year["y"]/4
print(ent_df_year.head(2))

In [ ]:
ent_sum = ent_df_year["y"].sum()
print("Entsoe sum in TWh", ent_sum/1e6)
print("Disaggregator sum in TWh", total_sum/1e6)
print("Disaggregator has", (total_sum/1e6) - (ent_sum/1e6), " more TWh than the entsoe.")

In [ ]:
ent_off = pd.read_csv("/mnt/data/oe215/rhindrikson/datasets/load/entsoe-hourly/data.csv")
ent_off["ds"] = pd.to_datetime(ent_off["ds"])
ent_off_year = ent_off[ent_off["ds"].dt.year ==year]
ent_off_year.head()

In [ ]:
fig = go.Figure()

df.index = pd.to_datetime(df.index)

# Add historical data
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["total"],
        mode="lines",
        name="Disaggregator"
    )
)

fig.add_trace(
    go.Scatter(
        x=ent_df_year["ds"],
        y=ent_df_year["y"],
        mode="lines",
        name="ENTSOE"
    ))

fig.add_trace(
    go.Scatter(
        x=ind_sum_df.index,
        y=ind_sum_df["total"],
        mode="lines",
        name="Industry"
    ))

fig.add_trace(
    go.Scatter(
        x=cts_sum_df.index,
        y=cts_sum_df["total"],
        mode="lines",
        name="CTS"
    ))

fig.add_trace(
    go.Scatter(
        x=hh_sum_df.index,
        y=hh_sum_df["total"],
        mode="lines",
        name="Households"
    ))

fig.update_layout(
    title=f'Demandregio Vs ENTSOE',
    xaxis_title='Date',
    yaxis_title='Load',
    hovermode='x unified',
    height=600
)

# fig.add_trace(
#     go.Scatter(
#         x=ent_off_year["ds"],
#         y=ent_off_year["y"],
#         mode="lines",
#         name="ENTSOE (Official)"
#     ))

fig.show(renderer='notebook')

# Load NHITS forecast for the first day of 2025

In [ ]:
y_hat_path ="/mnt/data/oe215/rhindrikson/el_load/ml_forecast/AutoNHITS_96_ts1096_vs7_HuberMQLoss_notemp_tr6/2026-02-09-22-45-54/forecast_df.csv"
y_hat_df = pd.read_csv(y_hat_path)
y_hat_df.head()

In [ ]:
fig.add_trace(
    go.Scatter(
        x=y_hat_df["ds"],
        y=y_hat_df["NHITS-median"] / 4,
        mode="lines",
        name="NHITS"
    )
)

fig.update_layout(
    xaxis=dict(
        range=['2024-12-20', '2025-01-02']
    ),
    title='ML Day-ahead forecast',
    xaxis_title='Date',
    yaxis_title='Load',
    hovermode='x unified',
    height=600
)
fig.show(renderer='notebook')

# Load forecasts for all days in 2022-2024

In [ ]:
# No refit
# path = "/mnt/data/oe215/rhindrikson/el_load/ml_forecast/20260113-114415/AutoNHITS_96_ts366_vs7_HuberMQLoss_notemp_tr5_y_hat_df.csv"
# With refit (takes 154.33 minutes to train)
path = "/mnt/data/oe215/rhindrikson/el_load/ml_forecast/AutoNHITS_96_ts1096_vs7_HuberMQLoss_notemp_tr6/2026-02-09-22-45-54/y_hat_df.csv"

ml_forecast = pd.read_csv(path)
ml_forecast.head(2)

In [ ]:
num_cutoffs = ml_forecast['cutoff'].nunique()
print(f"Number of cutoffs: {num_cutoffs}")

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# Add confidence intervals (from widest to narrowest)
# 100% interval
fig.add_trace(go.Scatter(
    x=ml_forecast['ds'],
    y=ml_forecast['AutoNHITS-hi-100'],
    mode='lines',
    name='100% CI Upper',
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=ml_forecast['ds'],
    y=ml_forecast['AutoNHITS-lo-100'],
    mode='lines',
    name='100% Confidence Interval',
    fill='tonexty',
    fillcolor='rgba(0, 100, 255, 0.1)',
    line=dict(width=0),
    showlegend=True,
    hoverinfo='skip'
))

# 90% interval
fig.add_trace(go.Scatter(
    x=ml_forecast['ds'],
    y=ml_forecast['AutoNHITS-hi-90'],
    mode='lines',
    name='90% CI Upper',
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=ml_forecast['ds'],
    y=ml_forecast['AutoNHITS-lo-90'],
    mode='lines',
    name='90% Confidence Interval',
    fill='tonexty',
    fillcolor='rgba(0, 100, 255, 0.2)',
    line=dict(width=0),
    showlegend=True,
    hoverinfo='skip'
))

# 80% interval
fig.add_trace(go.Scatter(
    x=ml_forecast['ds'],
    y=ml_forecast['AutoNHITS-hi-80'],
    mode='lines',
    name='80% CI Upper',
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=ml_forecast['ds'],
    y=ml_forecast['AutoNHITS-lo-80'],
    mode='lines',
    name='80% Confidence Interval',
    fill='tonexty',
    fillcolor='rgba(0, 100, 255, 0.3)',
    line=dict(width=0),
    showlegend=True,
    hoverinfo='skip'
))

# Plot prediction line
fig.add_trace(go.Scatter(
    x=ml_forecast['ds'],
    y=ml_forecast['AutoNHITS'],
    mode='lines',
    name='Predictions',
    line=dict(color='blue', width=2)
))

# Plot actual values
fig.add_trace(go.Scatter(
    x=ml_forecast['ds'],
    y=ml_forecast['y'],
    mode='lines',
    name='Actual',
    line=dict(color='orange', width=2)
))

fig.update_layout(
    title='AutoNHITS Predictions vs Actual Values (All Cutoffs)',
    xaxis_title='Date',
    yaxis_title='Load',
    hovermode='x unified',
    height=600
)

fig.show()